In [ ]:
# Imports
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import torchvision.transforms as transforms
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

In [ ]:
# Auto-detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = "https://github.com/your-username/your-repo.git"
    REPO_DIR = "/content/your-repo"
    
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}

    !cp -r "" /content/
    
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
else:
    src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
    sys.path.insert(0, src_path)

In [ ]:
from flow import Glow
from autoencoder import SpatialVAE

In [ ]:
# load data
data_file = "../latent_means_96.npy"
latent_means = np.load(data_file)
latent_means_tensor = torch.from_numpy(data['latents'].astype(np.float32))

latent_dataset = TensorDataset(all_latents)
train_size = int(0.9 * len(latent_dataset))
test_size  = len(latent_dataset) - train_size

train_split, test_split = random_split(latent_dataset, [train_size, test_size])

latent_train_loader = DataLoader(train_split, batch_size=64, shuffle=True,  num_workers=0)
latent_test_loader  = DataLoader(test_split,  batch_size=64, shuffle=False, num_workers=0)

In [ ]:
def train_glow(flow, train_loader, test_loader=None, epochs=200, lr=1e-4):
    device = next(flow.parameters()).device
    optimizer = torch.optim.Adam(flow.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

    for epoch in range(epochs):
        flow.train()
        total_nll = 0
        n_batches = 0

        for (mean,) in train_loader:
            mean = mean.to(device)
            log_prob = flow.log_prob(mean)
            loss = -log_prob.mean()

            if torch.isnan(loss):
                print("NaN detected, skipping batch")
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(flow.parameters(), 5.0)
            optimizer.step()

            total_nll += loss.item()
            n_batches += 1

        scheduler.step()

        val_str = ""
        if test_loader is not None:
            flow.eval()
            val_nll = 0
            val_batches = 0
            with torch.no_grad():
                for (mean,) in test_loader:
                    mean = mean.to(device)
                    log_prob = flow.log_prob(mean)
                    val_nll += (-log_prob.mean()).item()
                    val_batches += 1
            val_str = f" | Val NLL: {val_nll/val_batches:.2f}"

        avg_nll = total_nll / max(n_batches, 1)
        bpd = avg_nll / (1152 * np.log(2))
        print(f"Epoch {epoch+1}/{epochs} | "
              f"NLL: {avg_nll:.2f} | "
              f"BPD: {bpd:.3f}{val_str}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
autoencoder = torch.load('./spatial_autoencoder.pth')
autoencoder.to(device)
autoencoder.eval()

flow = Glow(
    in_channels=3,
    hidden_channels=128,
    K=6,
    n_scales=2,
).to(device)

print(f"Flow parameters: {sum(p.numel() for p in flow.parameters()):,}")
print("Training Flow")

train_glow(
    flow,
    train_loader=latent_train_loader,
    test_loader=latent_test_loader,
    epochs=50,
    lr=1e-3
)

torch.save(autoencoder, './flow.pth')